# Evaluating LLM Systems

Notebook for **Chapter 8** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 Why LLM Evaluation Is Hard, and the Layers of a Good Eval

## 8.2 Retrieval Metrics: recall@k, MRR and nDCG

In [ ]:
import numpy as np, math
ranks=[1,3,0,2]  # rank of gold, 0 = missing
print("recall@3",np.mean([0<r<=3 for r in ranks]),"MRR",round(np.mean([1/r if r else 0 for r in ranks]),4))
rel=[0,1,1,0]
dcg=sum(r/math.log2(i+2) for i,r in enumerate(rel))
idcg=sum(r/math.log2(i+2) for i,r in enumerate(sorted(rel,reverse=True)))
print("DCG",round(dcg,4),"IDCG",round(idcg,4),"nDCG",round(dcg/idcg,4))

recall@3 0.75 MRR 0.4583
DCG 1.1309 IDCG 1.6309 nDCG 0.6934


## 8.3 A Runnable Evaluation Harness

In [ ]:
import re
from rag import *
EVAL=[
 {"q":"How long does a refund take to process?","gold":"refund_policy.md","answer_has":"7 business days"},
 {"q":"Can I refund a gift card?","gold":"refund_policy.md","answer_has":"cannot be refunded"},
 {"q":"How many days does express shipping take?","gold":"shipping.md","answer_has":"2 days"},
 {"q":"Who pays customs fees on international orders?","gold":"shipping.md","answer_has":"buyer"},
 {"q":"When is phone support open?","gold":"support.md","answer_has":"9am to 6pm"},
 {"q":"How do I escalate a ticket?","gold":"support.md","answer_has":"reply to the ticket"},
 {"q":"What is the CEO's name?","gold":None,"answer_has":"I do not know"},
]
def faithful(answer,hits):
    """Cheap judge: the answer (minus citation) must appear verbatim in retrieved text."""
    text=" ".join(c[2] for c,_ in hits).replace("\n"," ")
    a=re.sub(r"\s*\[\d+\]","",answer).strip()
    return a=="I do not know." or a in text
def run(chunker,k):
    idx=build(chunker); rows=[]
    for ex in EVAL:
        hits=retrieve(idx,ex["q"],k); files=[h[0][0] for h in hits]
        ans=fake_llm(assemble(ex["q"],hits))
        rank=files.index(ex["gold"])+1 if ex["gold"] in files else 0
        rows.append({"q":ex["q"],
          "recall":(rank>0) if ex["gold"] else None,
          "rr":(1/rank if rank else 0) if ex["gold"] else None,
          "correct":ex["answer_has"].lower() in ans.lower(),
          "faithful":faithful(ans,hits),"ans":ans})
    return rows
def summarize(rows):
    r=[x for x in rows if x["recall"] is not None]
    return {"recall@k":round(sum(x["recall"] for x in r)/len(r),3),
            "MRR":round(sum(x["rr"] for x in r)/len(r),3),
            "correct":round(sum(x["correct"] for x in rows)/len(rows),3),
            "faithful":round(sum(x["faithful"] for x in rows)/len(rows),3)}
for name,ch in [("fixed12",chunk_fixed),("sections",chunk_sections)]:
    for k in (1,3):
        print(name,"k=",k,summarize(run(ch,k)))
print()
for x in run(chunk_fixed,3):
    if not x["correct"] or not x["faithful"]: print("FAIL:",x["q"],"->",x["ans"])

## 8.4 Faithfulness, LLM-as-Judge and Its Biases

In [ ]:
import numpy as np
human=[1,1,0,1,0,0,1,1,0,1]; judge=[1,1,0,1,1,0,1,1,1,1]
h,j=np.array(human),np.array(judge)
po=(h==j).mean(); pe=h.mean()*j.mean()+(1-h.mean())*(1-j.mean())
print("agree",po,"chance",round(pe,3),"kappa",round((po-pe)/(1-pe),3))
fp=((h==0)&(j==1)).sum(); print("false-pass",fp,"of",(h==0).sum())

agree 0.8 chance 0.56 kappa 0.545
false-pass 2 of 4


In [ ]:
import random
random.seed(1)
def mock_judge(a,b):
    if a["q"]==b["q"]: return "A" if random.random()<0.7 else "B"
    return "A" if a["q"]>b["q"] else "B"
pairs=[({"q":1,"n":"x"},{"q":1,"n":"y"}) for _ in range(100)]
print("x wins",sum(mock_judge(a,b)=="A" for a,b in pairs),"of 100")
consistent=0
for a,b in pairs:
    r1=mock_judge(a,b); r2=mock_judge(b,a)
    w1=a["n"] if r1=="A" else b["n"]; w2=b["n"] if r2=="A" else a["n"]
    consistent+= (w1==w2)
print("consistent winner after swap:",consistent,"of 100")

x wins 67 of 100
consistent winner after swap: 41 of 100


## 8.5 Statistics, Benchmarks and Online Monitoring

In [ ]:
import math
def wilson(k,n,z=1.96):
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); m=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return round((c-m)/d,3), round((c+m)/d,3)
for k,n in [(6,7),(60,70),(180,200),(900,1000)]:
    print(f"{k}/{n} = {k/n:.3f}  95% CI {wilson(k,n)}")

## 8.6 RAG Evaluation Frameworks, Judge Calibration and Confidence Intervals

In [ ]:
def ctx_precision(labels):
    hits, vals = 0, []
    for k, rel in enumerate(labels, 1):
        hits += rel
        if rel: vals.append(hits / k)
    return sum(vals) / len(vals) if vals else 0.0
print(round(ctx_precision([1, 0, 1, 0, 0]), 3))
print(round(ctx_precision([0, 0, 0, 1, 1]), 3))
print(round(ctx_precision([1, 1, 0, 0, 0]), 3))

0.833
0.325
1.0


In [ ]:
def kappa(both_pass, h_pass_j_fail, h_fail_j_pass, both_fail):
    n = both_pass + h_pass_j_fail + h_fail_j_pass + both_fail
    po = (both_pass + both_fail) / n
    hp = (both_pass + h_pass_j_fail) / n
    jp = (both_pass + h_fail_j_pass) / n
    pe = hp * jp + (1 - hp) * (1 - jp)
    return po, pe, (po - pe) / (1 - pe)
print("po %.3f pe %.3f kappa %.3f" % kappa(70, 5, 10, 15))
print("always-pass judge:  po %.3f pe %.3f kappa %.3f" % kappa(75, 0, 25, 0))

po 0.850 pe 0.650 kappa 0.571
always-pass judge:  po 0.750 pe 0.750 kappa 0.000


In [ ]:
import random
random.seed(3)
def biased_judge(first, second, quality):
    # true preference is weak; position bias gives "first" an extra 0.2 pull
    p_first = 0.5 + 0.5 * (quality[first] - quality[second]) + 0.2
    return first if random.random() < min(max(p_first, 0), 1) else second
quality = {"A": 0.60, "B": 0.55}                     # A is slightly better
single = sum(biased_judge("A", "B", quality) == "A" for _ in range(1000))
wins = ties = losses = 0
for _ in range(1000):
    r1 = biased_judge("A", "B", quality)
    r2 = biased_judge("B", "A", quality)
    if r1 == r2 == "A": wins += 1
    elif r1 == r2 == "B": losses += 1
    else: ties += 1
print("A first only: A wins", single, "of 1000 (looks like a landslide)")
print("with swap: A wins", wins, "ties", ties, "B wins", losses)

A first only: A wins 723 of 1000 (looks like a landslide)
with swap: A wins 245 ties 571 B wins 184


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
results = np.array([1] * 62 + [0] * 18)
boots = [rng.choice(results, size=80, replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boots, [2.5, 97.5])
print("accuracy", results.mean(), " 95% CI", round(lo, 3), round(hi, 3))
other = np.array([1] * 66 + [0] * 14)
diff = [rng.choice(other, 80).mean() - rng.choice(results, 80).mean() for _ in range(5000)]
print("new - old: mean", round(np.mean(diff), 3), " 95% CI", np.round(np.percentile(diff, [2.5, 97.5]), 3))

accuracy 0.775  95% CI 0.688 0.862
new - old: mean 0.05  95% CI [-0.075  0.175]


## Chapter recap

## Practice set

Problem 1 (easy). The gold chunk ranks for five questions are 1, 2, 1, 5 and missing. Compute recall@3 and MRR.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch08.html